In [0]:
def get_tables_needing_ownership_change(table_owner: str) -> list:
    """
    This method returns a list of all fully qualified dev tables in Unity Catalog, besides those
    in information schema, that require an update to ownership. Tables require an ownership update
    if they aren't already owned by the specified owner.

    Args:
        table_owner: The desired owner of the table.

    Returns:
        uc_tables: The list of dev tables in UC that require an ownership change
                   and aren't part of the information schema.
    """

    uc_tables = []

    # only include dev catalogs, prod catalog ownership should not be changed from skynet
    catalogs = [c[0] for c in spark.sql("show catalogs like '*_dev'").select("catalog").collect()]
    for catalog in catalogs:
        # skip information schema tables and ones already owned by data maintainers
        tables_to_change = [
            t[0]
            for t in spark.sql(
                f"""
        select concat(table_catalog, '.', table_schema, '.', table_name) as fully_qualified_table_name
        from {catalog}.information_schema.tables
        where table_owner <> '{table_owner}'
        and table_schema <> 'information_schema'
        """
            )
            .select("fully_qualified_table_name")
            .collect()
        ]

        uc_tables.extend(tables_to_change)

    return uc_tables


In [0]:

def assign_uc_table_owner(uc_tables: list, table_owner: str) -> None:
    """
    This method assigns a UC table to a specified owner. Currently this is set to naz-beertech-data-maintainers
    so that engineers can delete tables initially created by other engineers.

    Args:
        uc_tables: The list of UC tables in which to assign ownership to naz-beertech-data-maintainers
        table_owner: The desired owner of the table.

    Returns:
        None
    """

    for table in uc_tables:
        # set the table to the proper owner
        spark.sql(f"ALTER TABLE {table} OWNER TO `{table_owner}`")
        print(f"Set owner on table {table} to {table_owner}")


In [0]:
DATA_MAINTAINERS_GROUP = "naz-beertech-data-maintainers"
uc_tables = get_tables_needing_ownership_change(table_owner=DATA_MAINTAINERS_GROUP)
if len(uc_tables) == 0:
    print(f"No tables require ownership change, all tables are owned by {DATA_MAINTAINERS_GROUP}")
else:
    assign_uc_table_owner(uc_tables=uc_tables, table_owner=DATA_MAINTAINERS_GROUP)